In [1]:
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [2]:
print("Starting Extraction Phase...")

fact_df = pd.read_csv("FACT_SALES.csv")
dim_date_df = pd.read_csv("DIM_DATE.csv")
dim_product_df = pd.read_csv("DIM_PRODUCT.csv")
dim_customer_df = pd.read_csv("DIM_CUSTOMER.csv")

print("All CSV files loaded successfully.")

Starting Extraction Phase...
All CSV files loaded successfully.


In [3]:
print("Starting Transformation Phase...")

# Standardize column names
for df in [fact_df, dim_date_df, dim_product_df, dim_customer_df]:
    df.columns = df.columns.str.upper().str.replace(" ", "_")

# Convert ORDER_DATE
fact_df["ORDER_DATE"] = pd.to_datetime(fact_df["ORDER_DATE"])
fact_df["ORDER_DATE"] = fact_df["ORDER_DATE"].dt.strftime("%Y-%m-%d")

# Ensure numeric fields
fact_df["SALES"] = fact_df["SALES"].astype(float)

print("Transformation Completed Successfully")

Starting Transformation Phase...
Transformation Completed Successfully


In [4]:
conn = snowflake.connector.connect(
    user="YOUR_USERNAME",
    password="YOUR_PASSWORD",
    account="YOUR_ACCOUNT",
    warehouse="RETAIL_WH",
    database="RETAIL_DW",
    schema="RETAIL"
)

print("Connection block provided. Credentials removed for security.")

#  Snowflake Connection

#  Snowflake credentials have been removed for security reasons.
#  Please configure your own Snowflake account credentials before execution.


# SNOWFLAKE CONFIGURATION INSTRUCTIONS
# -------------------------------------------------------
# Before running this notebook, set the following
# environment variables in your system:

# For Windows (PowerShell):
# setx SNOWFLAKE_USER "your_username"
# setx SNOWFLAKE_PASSWORD "your_password"
# setx SNOWFLAKE_ACCOUNT "your_account"

# For Mac/Linux:
# export SNOWFLAKE_USER=your_username
# export SNOWFLAKE_PASSWORD=your_password
# export SNOWFLAKE_ACCOUNT=your_account

# After setting environment variables, restart Jupyter
# and run the notebook.


Connected to Snowflake Successfully


In [5]:
print("Starting Load Phase...")

write_pandas(conn, dim_date_df, "DIM_DATE")
write_pandas(conn, dim_product_df, "DIM_PRODUCT")
write_pandas(conn, dim_customer_df, "DIM_CUSTOMER")
write_pandas(conn, fact_df, "FACT_SALES")

print("All tables loaded successfully!")

Starting Load Phase...
All tables loaded successfully!


In [6]:
cursor = conn.cursor()

print("FACT_SALES in Snowflake:")
cursor.execute("SELECT * FROM FACT_SALES LIMIT 5")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

print("DIM_DATE in Snowflake:")
cursor.execute("SELECT * FROM DIM_DATE LIMIT 5")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

FACT_SALES in Snowflake:


,ORDER_ID,CUSTOMER_ID,PRODUCT_ID,ORDER_DATE,SALES
0,CA-2017-152156,CG-12520,FUR-BO-10001798,2017-11-08,261.96
1,CA-2017-152156,CG-12520,FUR-CH-10000454,2017-11-08,731.94
2,CA-2017-138688,DV-13045,OFF-LA-10000240,2017-06-12,14.62
3,US-2016-108966,SO-20335,FUR-TA-10000577,2016-10-11,957.58
4,US-2016-108966,SO-20335,OFF-ST-10000760,2016-10-11,22.37


DIM_DATE in Snowflake:


,DATE_KEY,YEAR,QUARTER,MONTH,MONTH_NAME,DAY,DAY_OF_WEEK
0,2016-12-27,2016,4,12,Dec,27,Tue
1,2018-09-19,2018,3,9,Sep,19,Wed
2,2018-12-09,2018,4,12,Dec,9,Sun
3,2017-04-05,2017,2,4,Apr,5,Wed
4,2018-12-22,2018,4,12,Dec,22,Sat


In [7]:
conn.close()
print("ETL Pipeline Completed Successfully.")

ETL Pipeline Completed Successfully.
